
# Caderno de Trading — LULU, GAP (GPS), ANF — Yahoo Finance  
**ADX/ATR, filtros EMA50/200, Alarmes + Notificações (E-mail/Webhook)**

*Autor:* Engenheiro de dados / trader quantitativo  
*Data:* **2025-10-04**

**Novidades:** envio opcional de **notificações por e-mail e webhook** quando um alarme é detectado (Golden/Death Cross, ADX&gt;25 e subindo, ATR crescente). Inclui **log** simples para evitar alertas duplicados.

> **Aviso**: Conteúdo educacional; **não** é recomendação. Configure e teste em ambiente seguro.



## 0) Dependências
Execute para garantir os pacotes necessários.


In [ ]:

import sys, subprocess

def _pip_install(pkg: str):
    try:
        __import__(pkg.split("==")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

for p in [
    "yfinance>=0.2.40",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "mplfinance>=0.12.10b0",
    "pytz",
    "numpy>=1.24.0",
    "requests>=2.31.0"
]:
    _pip_install(p)

print("Dependências OK.")



## 1) Configurações de Notificação (EDITE AQUI)
- Defina **EMAIL\_ENABLED** e/ou **WEBHOOK\_ENABLED** como `True` para ativar.
- Preencha seu servidor SMTP, credenciais e destinatários (use variáveis de ambiente se preferir).
- Para **WEBHOOK\_URL**, você pode usar Slack/Discord/Teams ou um endpoint próprio (JSON via POST).


In [ ]:

import os

# ====== ATIVAÇÃO ======
EMAIL_ENABLED = False
WEBHOOK_ENABLED = False

# ====== E-MAIL (SMTP) ======
SMTP_SERVER   = os.getenv("SMTP_SERVER", "smtp.seuservidor.com")
SMTP_PORT     = int(os.getenv("SMTP_PORT", "587"))
SMTP_USER     = os.getenv("SMTP_USER", "usuario")
SMTP_PASSWORD = os.getenv("SMTP_PASSWORD", "senha")
SMTP_USE_TLS  = True
SMTP_FROM     = os.getenv("SMTP_FROM", "alertas@seudominio.com")
EMAIL_TO      = os.getenv("EMAIL_TO", "voce@seudominio.com")  # um ou vários separados por vírgula

# ====== WEBHOOK ======
WEBHOOK_URL     = os.getenv("WEBHOOK_URL", "https://exemplo.com/webhook")
WEBHOOK_HEADERS = {"Content-Type": "application/json"}  # ajuste se necessário

# ====== LOG ======
ALERT_LOG_PATH = "alert_log.csv"  # arquivo local p/ evitar duplicatas



## 2) Imports, utilitários e indicadores
Inclui **ADX**, **ATR**, **EMA/RSI/MACD**, detecção de **Golden/Death Cross** e utilitários de notificação/log.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import mplfinance as mpf
import pytz, json, csv, hashlib, smtplib
from email.message import EmailMessage
import requests
from datetime import datetime, timedelta
from pathlib import Path

# ---------- Parâmetros ----------
TICKERS_INPUT = ["LULU", "GAP", "ANF"]
TICKER_MAP = {"GAP": "GPS"}  # mapeia "GAP" -> "GPS" (ticker oficial)
LOCAL_TZ = "America/Sao_Paulo"

INTERVALO_POR_PERIODO = {
    "3mo": "1d",
    "1mo": "1d",
    "7d":  "60m",
    "1d":  "30m",
}

# ---------- Indicadores ----------
def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()

def rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    up = np.where(delta > 0, delta, 0.0)
    down = np.where(delta < 0, -delta, 0.0)
    roll_up = pd.Series(up, index=series.index).ewm(alpha=1/period, adjust=False).mean()
    roll_down = pd.Series(down, index=series.index).ewm(alpha=1/period, adjust=False).mean()
    rs = roll_up / (roll_down.replace(0, np.nan))
    rsi_vals = 100 - (100 / (1 + rs))
    return rsi_vals.fillna(50.0)

def macd(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    ema_fast = ema(series, fast)
    ema_slow = ema(series, slow)
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    hist = macd_line - signal_line
    return macd_line, signal_line, hist

def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr

def atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    tr = true_range(high, low, close)
    return tr.ewm(alpha=1/period, adjust=False).mean()

def adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    tr = true_range(high, low, close)
    atr_vals = tr.ewm(alpha=1/period, adjust=False).mean()

    plus_di = 100 * (pd.Series(plus_dm, index=high.index).ewm(alpha=1/period, adjust=False).mean() / atr_vals)
    minus_di = 100 * (pd.Series(minus_dm, index=high.index).ewm(alpha=1/period, adjust=False).mean() / atr_vals)

    dx = (100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)).fillna(0)
    adx_vals = dx.ewm(alpha=1/period, adjust=False).mean()
    return adx_vals

# ---------- Helpers ----------
def crossover_up(a: pd.Series, b: pd.Series) -> pd.Series:
    return (a.shift(1) <= b.shift(1)) & (a > b)

def crossover_down(a: pd.Series, b: pd.Series) -> pd.Series:
    return (a.shift(1) >= b.shift(1)) & (a < b)

def baixar_dados(ticker: str, period: str, interval: str) -> pd.DataFrame:
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=False, progress=False)
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise RuntimeError(f"Sem dados para {ticker} em period={period}, interval={interval}.")
    df = df.dropna().copy()
    if not df.index.tz:
        df.index = df.index.tz_localize("UTC")
    return df

def fuso_horario(df: pd.DataFrame, tz: str) -> pd.DataFrame:
    return df.tz_convert(tz)

def resolve_ticker(t: str) -> str:
    return TICKER_MAP.get(t, t)

# ---------- Plotters (sem estilos/cores específicas) ----------
def plot_preco_simples(df: pd.DataFrame, titulo: str = ""):
    plt.figure(figsize=(10, 4))
    plt.plot(df.index, df["Close"], label="Close")
    plt.title(titulo if titulo else "Preço (Close)")
    plt.xlabel("Tempo")
    plt.ylabel("Preço")
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_candles(df: pd.DataFrame, titulo: str = ""):
    mpf.plot(df, type="candle", style="yahoo", title=titulo, volume=False)

def plot_macd(df: pd.DataFrame, titulo: str = ""):
    close = df["Close"]
    macd_line, signal_line, hist = macd(close)
    plt.figure(figsize=(10, 4))
    plt.plot(macd_line.index, macd_line, label="MACD")
    plt.plot(signal_line.index, signal_line, label="Signal")
    buys  = crossover_up(macd_line, signal_line)
    sells = crossover_down(macd_line, signal_line)
    plt.scatter(macd_line.index[buys], macd_line[buys], marker="^", label="Buy")
    plt.scatter(macd_line.index[sells], macd_line[sells], marker="v", label="Sell")
    plt.title(titulo if titulo else "MACD")
    plt.xlabel("Tempo")
    plt.ylabel("Valor")
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_macd_filtrado(df: pd.DataFrame, ema50: pd.Series, ema200: pd.Series, titulo: str = ""):
    close = df["Close"]
    macd_line, signal_line, hist = macd(close)
    bull = (close > ema50) & (ema50 > ema200)
    bear = (close < ema50) & (ema50 < ema200)
    buys_raw  = crossover_up(macd_line, signal_line)
    sells_raw = crossover_down(macd_line, signal_line)
    buys_f  = buys_raw & bull
    sells_f = sells_raw & bear

    plt.figure(figsize=(10, 4))
    plt.plot(macd_line.index, macd_line, label="MACD")
    plt.plot(signal_line.index, signal_line, label="Signal")
    plt.scatter(macd_line.index[buys_f], macd_line[buys_f], marker="^", label="Buy (filtro EMA50/200)")
    plt.scatter(macd_line.index[sells_f], macd_line[sells_f], marker="v", label="Sell (filtro EMA50/200)")
    plt.title(titulo if titulo else "MACD (filtrado por EMA50/200)")
    plt.xlabel("Tempo")
    plt.ylabel("Valor")
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_rsi(df: pd.DataFrame, titulo: str = ""):
    r = rsi(df["Close"])
    plt.figure(figsize=(10, 3))
    plt.plot(r.index, r, label="RSI(14)")
    plt.axhline(70, linestyle="--")
    plt.axhline(30, linestyle="--")
    overbought = (r.shift(1) < 70) & (r >= 70)
    oversold   = (r.shift(1) > 30) & (r <= 30)
    plt.scatter(r.index[overbought], r[overbought], marker="x", label="Overbought")
    plt.scatter(r.index[oversold],   r[oversold],   marker="x", label="Oversold")
    plt.title(titulo if titulo else "RSI(14)")
    plt.xlabel("Tempo")
    plt.ylabel("RSI")
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_ema21_com_sinais(df: pd.DataFrame, titulo: str = ""):
    px = df.copy()
    px["EMA21"] = ema(px["Close"], 21)
    ent = crossover_up(px["Close"], px["EMA21"])
    sai = crossover_down(px["Close"], px["EMA21"])
    plt.figure(figsize=(11, 5))
    plt.plot(px.index, px["Close"], label="Close")
    plt.plot(px.index, px["EMA21"], label="EMA21")
    plt.scatter(px.index[ent], px["Close"][ent], marker="^", label="Entrada (cross up)")
    plt.scatter(px.index[sai], px["Close"][sai], marker="v", label="Saída (cross down)")
    plt.title(titulo if titulo else "EMA21 - Entradas/Saídas")
    plt.xlabel("Tempo")
    plt.ylabel("Preço")
    plt.grid(True)
    plt.legend()
    plt.show()

def plot_ema50200(df: pd.DataFrame, titulo: str = ""):
    px = df.copy()
    px["EMA50"]  = ema(px["Close"], 50)
    px["EMA200"] = ema(px["Close"], 200)
    golden = crossover_up(px["EMA50"], px["EMA200"])
    death  = crossover_down(px["EMA50"], px["EMA200"])

    plt.figure(figsize=(11, 5))
    plt.plot(px.index, px["Close"], label="Close")
    plt.plot(px.index, px["EMA50"], label="EMA50")
    plt.plot(px.index, px["EMA200"], label="EMA200")
    plt.scatter(px.index[golden], px["EMA50"][golden], marker="^", label="Golden Cross (50>200)")
    plt.scatter(px.index[death],  px["EMA50"][death],  marker="v", label="Death Cross (50<200)")
    plt.title(titulo if titulo else "EMA50/EMA200 — Regime e Crossovers")
    plt.xlabel("Tempo")
    plt.ylabel("Preço")
    plt.grid(True)
    plt.legend()
    plt.show()
    return px["EMA50"], px["EMA200"], golden, death

def plot_adx(df: pd.DataFrame, titulo: str = ""):
    a = adx(df["High"], df["Low"], df["Close"], period=14)
    plt.figure(figsize=(10, 3))
    plt.plot(a.index, a, label="ADX(14)")
    plt.axhline(25, linestyle="--")
    plt.title(titulo if titulo else "ADX(14) — Força da Tendência")
    plt.xlabel("Tempo")
    plt.ylabel("ADX")
    plt.grid(True)
    plt.legend()
    plt.show()
    return a

def plot_atr(df: pd.DataFrame, titulo: str = ""):
    a = atr(df["High"], df["Low"], df["Close"], period=14)
    plt.figure(figsize=(10, 3))
    plt.plot(a.index, a, label="ATR(14)")
    plt.title(titulo if titulo else "ATR(14) — Volatilidade")
    plt.xlabel("Tempo")
    plt.ylabel("ATR")
    plt.grid(True)
    plt.legend()
    plt.show()
    return a

# ---------- Notificações & Log ----------
def _alert_key(ticker: str, evento: str, quando) -> str:
    raw = f"{ticker}|{evento}|{str(quando)}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def _is_duplicate(key: str, path: str) -> bool:
    p = Path(path)
    if not p.exists():
        return False
    try:
        with p.open("r", newline="", encoding="utf-8") as f:
            r = csv.reader(f)
            for row in r:
                if row and row[0] == key:
                    return True
    except Exception:
        return False
    return False

def _append_log(key: str, path: str):
    p = Path(path)
    try:
        with p.open("a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow([key, datetime.utcnow().isoformat()])
    except Exception as e:
        print("Falha ao registrar log:", e)

def enviar_email(assunto: str, corpo: str):
    if not EMAIL_ENABLED:
        print("[EMAIL desabilitado] Preview:", assunto)
        print(corpo)
        return
    msg = EmailMessage()
    msg["Subject"] = assunto
    msg["From"] = SMTP_FROM
    msg["To"] = EMAIL_TO
    msg.set_content(corpo)
    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            if SMTP_USE_TLS:
                server.starttls()
            if SMTP_USER:
                server.login(SMTP_USER, SMTP_PASSWORD)
            server.send_message(msg)
        print("E-mail enviado.")
    except Exception as e:
        print("Falha ao enviar e-mail:", e)

def enviar_webhook(payload: dict):
    if not WEBHOOK_ENABLED:
        print("[WEBHOOK desabilitado] Preview payload:", json.dumps(payload, ensure_ascii=False))
        return
    try:
        r = requests.post(WEBHOOK_URL, headers=WEBHOOK_HEADERS, data=json.dumps(payload))
        print("Webhook status:", r.status_code, r.text[:200])
    except Exception as e:
        print("Falha ao enviar webhook:", e)

def notificar_alerta(row, extra: dict):
    # row: {"ticker","evento","quando","motivo"}
    key = _alert_key(row["ticker"], row["evento"], row["quando"])
    if _is_duplicate(key, ALERT_LOG_PATH):
        print("Alerta duplicado ignorado:", row)
        return
    assunto = f"[ALERTA] {row['ticker']} — {row['evento']}"
    corpo = f"""Ticker: {row['ticker']}
Evento: {row['evento']}
Quando: {row['quando']}
Motivo: {row['motivo']}

Resumo:
- Regime: {extra.get('regime')}
- ADX(14): {extra.get('adx')}
- ATR(14): {extra.get('atr')}
- Preço: {extra.get('preco')}
"""
    enviar_email(assunto, corpo)
    payload = {
        "type": "alerta_trading",
        "ticker": row["ticker"],
        "evento": row["evento"],
        "quando": str(row["quando"]),
        "motivo": row["motivo"],
        "resumo": extra
    }
    enviar_webhook(payload)
    _append_log(key, ALERT_LOG_PATH)

# ---------- Regras e sizing ----------
def regime_status(close: pd.Series, ema50: pd.Series, ema200: pd.Series) -> str:
    if close.iloc[-1] > ema50.iloc[-1] > ema200.iloc[-1]:
        return "Bullish (Close>EMA50>EMA200)"
    if close.iloc[-1] < ema50.iloc[-1] < ema200.iloc[-1]:
        return "Bearish (Close<EMA50<EMA200)"
    return "Neutro/Transição"

def gerar_alertas(ticker: str, df3m: pd.DataFrame, ema50: pd.Series, ema200: pd.Series, adx3m: pd.Series, atr3m: pd.Series) -> pd.DataFrame:
    alerts = []
    golden_now = crossover_up(ema50, ema200)
    death_now  = crossover_down(ema50, ema200)

    if golden_now.iloc[-1]:
        alerts.append({
            "ticker": ticker, "evento": "Golden Cross",
            "quando": df3m.index[-1], "motivo": "Mudança potencial para regime bullish (EMA50 cruzou acima da EMA200)."
        })
    if death_now.iloc[-1]:
        alerts.append({
            "ticker": ticker, "evento": "Death Cross",
            "quando": df3m.index[-1], "motivo": "Mudança potencial para regime bearish (EMA50 cruzou abaixo da EMA200)."
        })
    if len(adx3m) > 2 and (adx3m.iloc[-1] > 25) and (adx3m.iloc[-1] > adx3m.iloc[-2]):
        alerts.append({
            "ticker": ticker, "evento": "ADX Forte/Subindo",
            "quando": df3m.index[-1], "motivo": "Força de tendência aumentando (ADX>25 e crescente)."
        })
    if len(atr3m) > 2 and (atr3m.iloc[-1] > atr3m.iloc[-2] > atr3m.iloc[-3]):
        alerts.append({
            "ticker": ticker, "evento": "ATR Crescente",
            "quando": df3m.index[-1], "motivo": "Volatilidade em alta; considerar stop mais largo e reduzir tamanho da posição."
        })
    return pd.DataFrame(alerts)

def sugerir_tamanho_posicao(preco_atual: float, atr_val: float, equity: float = 100000.0, risco_pct: float = 0.01, atr_mult_stop: float = 2.0):
    risco_valor = equity * risco_pct
    stop_dist = atr_mult_stop * atr_val
    if stop_dist <= 0:
        return 0.0, risco_valor, stop_dist
    qty = max(risco_valor / stop_dist, 0.0)
    return qty, risco_valor, stop_dist



## 3) Por que os **alarmes**? O que são **Golden/Death Cross**?
- **Golden Cross**: **EMA50 cruza para cima a EMA200** → sinal de **possível regime bullish**. Útil para capturar inícios/continuações de tendência de alta.
- **Death Cross**: **EMA50 cruza para baixo a EMA200** → **possível regime bearish**. Útil para reduzir risco/exposição ou avaliar vendas quando permitido.
- **ADX** mede **força** (não direção) da tendência; **>25 e subindo** sugere fortalecimento do movimento.
- **ATR** reflete **volatilidade**; **ATR↑** sugere usar **stops mais largos** e **tamanho de posição menor** para manter risco constante.
- Alarmes garantem **disciplina e reação rápida** a mudanças de **regime, força e volatilidade**.



## 4) Função principal por ticker (gera gráficos, sinais e envia notificações)


In [ ]:

def analisar_ticker(ticker_in: str, equity: float = 100000.0, risco_pct: float = 0.01, atr_mult_stop: float = 2.0):
    disp = ticker_in
    tk   = resolve_ticker(ticker_in)

    print(f"\n=== {disp} (Yahoo: {tk}) ===")

    # Downloads
    dados_3m = baixar_dados(tk, "3mo", INTERVALO_POR_PERIODO["3mo"])
    dados_1m = baixar_dados(tk, "1mo", INTERVALO_POR_PERIODO["1mo"])
    dados_7d = baixar_dados(tk, "7d",  INTERVALO_POR_PERIODO["7d"])
    dados_1d = baixar_dados(tk, "1d",  INTERVALO_POR_PERIODO["1d"])

    # 24h candles 1h local
    dados_2d_1h = baixar_dados(tk, "2d", "60m")
    dados_2d_1h_local = fuso_horario(dados_2d_1h, LOCAL_TZ)
    corte = dados_2d_1h_local.index.max() - pd.Timedelta("24H")
    ult_24h = dados_2d_1h_local.loc[dados_2d_1h_local.index > corte].copy()

    # Converte intraday
    dados_7d_local = fuso_horario(dados_7d, LOCAL_TZ)
    dados_1d_local = fuso_horario(dados_1d, LOCAL_TZ)

    # Gráficos principais
    plot_preco_simples(dados_3m, titulo=f"{disp} — Close (últimos 3 meses)")
    if len(ult_24h) < 3:
        print("Poucos candles disponíveis nas últimas 24h (mercado pode ter estado fechado). Exibindo o que houver.")
    plot_candles(ult_24h, titulo=f"{disp} — Candles 1h (últimas 24h) [TZ: {LOCAL_TZ}]")

    # MACD bruto
    plot_macd(dados_3m, titulo=f"{disp} — MACD (3M, 1d)")
    plot_macd(dados_1m, titulo=f"{disp} — MACD (1M, 1d)")
    plot_macd(dados_7d_local, titulo=f"{disp} — MACD (7D, 60m)")
    plot_macd(dados_1d_local, titulo=f"{disp} — MACD (1D, 30m)")

    # EMA50/200 e MACD filtrado
    ema50_3m, ema200_3m, golden_mask, death_mask = plot_ema50200(dados_3m, titulo=f"{disp} — EMA50/EMA200 (3M, 1d)")
    plot_macd_filtrado(dados_3m, ema50_3m, ema200_3m, titulo=f"{disp} — MACD (3M) filtrado por EMA50/200")

    # RSI (multijanela)
    plot_rsi(dados_3m, titulo=f"{disp} — RSI(14) (3M, 1d)")
    plot_rsi(dados_1m, titulo=f"{disp} — RSI(14) (1M, 1d)")
    plot_rsi(dados_7d_local, titulo=f"{disp} — RSI(14) (7D, 60m)")
    plot_rsi(dados_1d_local, titulo=f"{disp} — RSI(14) (1D, 30m)")

    # EMA21 entradas/saídas (3M)
    plot_ema21_com_sinais(dados_3m, titulo=f"{disp} — EMA21 Entradas/Saídas (3M, 1d)")

    # ADX & ATR (3M)
    adx3m = plot_adx(dados_3m, titulo=f"{disp} — ADX(14) (3M, 1d)")
    atr3m = plot_atr(dados_3m, titulo=f"{disp} — ATR(14) (3M, 1d)")

    # Painel resumido
    macd_line_3m, signal_3m, _ = macd(dados_3m["Close"])
    buys_raw  = crossover_up(macd_line_3m, signal_3m)
    sells_raw = crossover_down(macd_line_3m, signal_3m)
    bull = (dados_3m["Close"] > ema50_3m) & (ema50_3m > ema200_3m)
    bear = (dados_3m["Close"] < ema50_3m) & (ema50_3m < ema200_3m)
    buys_f = buys_raw & bull
    sells_f = sells_raw & bear

    def last_idx(mask: pd.Series):
        return mask.index[mask].max() if mask.any() else None

    last_buy_raw  = last_idx(buys_raw)
    last_sell_raw = last_idx(sells_raw)
    last_buy_f    = last_idx(buys_f)
    last_sell_f   = last_idx(sells_f)

    resumo = []
    regime_txt = "Bullish (Close>EMA50>EMA200)" if bull.iloc[-1] else ("Bearish (Close<EMA50<EMA200)" if bear.iloc[-1] else "Neutro/Transição")
    resumo.append(("Regime (3M)", regime_txt))
    resumo.append(("MACD 3M (bruto)",  f"BUY @ {last_buy_raw}" if last_buy_raw else "—"))
    resumo.append(("MACD 3M (bruto)",  f"SELL @ {last_sell_raw}" if last_sell_raw else "—"))
    resumo.append(("MACD 3M (filtrado)", f"BUY @ {last_buy_f}"   if last_buy_f   else "—"))
    resumo.append(("MACD 3M (filtrado)", f"SELL @ {last_sell_f}" if last_sell_f  else "—"))
    # RSI status
    def status_rsi(close_series):
        r = rsi(close_series)
        last = r.iloc[-1]
        if last >= 70: return f"Overbought (RSI={last:.1f})"
        if last <= 30: return f"Oversold (RSI={last:.1f})"
        if last >= 60: return f"Tendência forte (RSI={last:.1f})"
        if last <= 40: return f"Tendência fraca/pressão vendedora (RSI={last:.1f})"
        return f"Neutro (RSI={last:.1f})"
    resumo.append(("RSI 3M", status_rsi(dados_3m["Close"])))
    resumo.append(("ADX 3M", f"{adx3m.iloc[-1]:.1f}"))
    resumo.append(("ATR 3M", f"{atr3m.iloc[-1]:.2f}"))

    display(pd.DataFrame(resumo, columns=["Métrica", "Status"]))

    # Geração de alarmes + envio de notificações
    alerts_df = gerar_alertas(disp, dados_3m, ema50_3m, ema200_3m, adx3m, atr3m)
    if alerts_df.empty:
        print("Sem alarmes no candle mais recente.")
    else:
        print("ALARMES:")
        display(alerts_df)

        # Notificações (opcional)
        for _, row in alerts_df.iterrows():
            extra = {
                "regime": regime_txt,
                "adx": float(adx3m.iloc[-1]),
                "atr": float(atr3m.iloc[-1]),
                "preco": float(dados_3m['Close'].iloc[-1])
            }
            notificar_alerta(row, extra)

    # Sizing por ATR (exemplo)
    preco_atual = float(dados_3m["Close"].iloc[-1])
    atr_val = float(atr3m.iloc[-1])
    qty, risco_valor, stop_dist = sugerir_tamanho_posicao(preco_atual, atr_val, equity=equity, risco_pct=risco_pct, atr_mult_stop=atr_mult_stop)
    sizing_info = pd.DataFrame([{
        "Ticker": disp,
        "Preço Atual": round(preco_atual, 2),
        "ATR(14)": round(atr_val, 3),
        "ATR Mult Stop": atr_mult_stop,
        "Distância Stop": round(stop_dist, 3),
        "Risco por Trade": round(risco_valor, 2),
        "Qtd. sugerida (ação aprox.)": int(qty)
    }])
    print("Sizing por ATR (exemplo paramétrico):")
    display(sizing_info)



## 5) Executar análise (ajuste parâmetros de risco conforme sua política)
- **Ative** EMAIL/WEBHOOK na seção de **Configurações** antes de rodar este bloco, se quiser receber alertas.
- Um **log local** (`alert_log.csv`) evita disparos duplicados do mesmo evento.


In [ ]:

EQUITY = 100000.0   # capital de referência
RISCO_PCT = 0.01    # 1% de risco por trade
ATR_MULT = 2.0      # stop a 2x ATR

for tk in TICKERS_INPUT:
    analisar_ticker(tk, equity=EQUITY, risco_pct=RISCO_PCT, atr_mult_stop=ATR_MULT)



## 6) Dicas de integração rápida de webhook
- **Slack**: crie um *Incoming Webhook* (formato JSON). Você pode mapear `payload["text"]` no `enviar_webhook`.
- **Discord**: use a URL de webhook do canal; aceita JSON e *embeds*.
- **Teams**: use *Incoming Webhook* no conector do canal.
- **Custom API**: ajuste cabeçalhos e formato conforme sua API.
